# 공공데이터 API → DuckDB 적재

교통수단 조회 API부터 시작. 추후 환승시간, 노인 비율 등 테이블 추가 예정.

In [ ]:
# ============================================================
# 셀 1 - 환경 설정 + DB 헬퍼 정의
# ============================================================
# ⚠️ 락 점유 최소화 패턴:
#   - 커넥션을 전역(con)으로 오래 유지하지 않음
#   - 각 작업마다 db_open() 컨텍스트로 짧게 열고 즉시 닫음
#   - try/finally 보장 → 셀 실행 중 예외가 나도 커넥션 반드시 해제
#   - 결과적으로 여러 노트북을 동시에 열어도 락 충돌 없음

import os
import time
import requests
import duckdb
import pandas as pd
from pathlib import Path
from contextlib import contextmanager
from urllib.parse import unquote
from dotenv import load_dotenv

load_dotenv()
SERVICE_KEY = unquote(os.environ["MY_SERVICE_KEY"])

DB_PATH = "data/seoul.duckdb"
Path("data").mkdir(exist_ok=True)


@contextmanager
def db_open(read_only: bool = False):
    """짧게 연결하고 무조건 닫는 컨텍스트 매니저.

    - read_only=True 면 여러 커넥션이 동시에 열려도 됨 (읽기 전용)
    - read_only=False (default) 는 쓰기 가능하지만 락 점유 → 짧게 쓰고 끝내기
    - try/finally 로 예외 발생 시에도 con.close() 보장
    """
    con = duckdb.connect(DB_PATH, read_only=read_only)
    try:
        yield con
    finally:
        con.close()


print(f"SERVICE_KEY 로드됨 (길이 {len(SERVICE_KEY)})")
print(f"DB: {DB_PATH} (헬퍼 준비됨, 아직 연결 안 함)")

In [32]:
# ============================================================
# 셀 1-1 - 공공데이터 표준 응답 Pydantic 모델 (제네릭)
# ============================================================
# 공공데이터포털 API 대부분은 아래와 같은 공통 JSON 구조로 응답함:
#
#   {
#     "Response": {
#       "header": { "resultCode": "200", "resultMsg": "SUCCESS" },
#       "body": {
#         "items": {
#           "item": [ { ... }, { ... } ]   ← 실제 데이터 배열
#         },
#         "pageNo": 1, "numOfRows": 1000, "totalCount": 1959
#       }
#     }
#   }
#
# 이 공통 껍데기를 Generic[T]로 한 번만 정의하고, item 모델 T만 교체해서 재사용.
# (예: PublicDataResponse[TrfcMnsItem] / PublicDataResponse[CardTypeItem] ...)
#
# 장점:
#   1) 모든 API에서 동일한 헤더/페이지네이션 필드 접근 방식 공유
#   2) API 스키마가 바뀌면 Pydantic validation이 즉시 실패 → 조기 감지
#   3) item이 1건일 땐 dict, 여러건일 땐 list로 오는 공공데이터 API 변동을 validator로 흡수

from typing import Generic, TypeVar
from pydantic import BaseModel, field_validator

# TypeVar: 제네릭의 타입 파라미터. T는 반드시 BaseModel의 서브클래스여야 함.
T = TypeVar("T", bound=BaseModel)


class Header(BaseModel):
    """응답 헤더. resultCode '200' 또는 '00'이 정상."""
    resultCode: str
    resultMsg: str


class Items(BaseModel, Generic[T]):
    """items 래퍼. 실제 데이터 리스트는 items.item에 들어 있음."""
    item: list[T] = []   # default=[] → items 키가 아예 없을 때 대비

    # ⚠️ 공공데이터 API의 함정: item이 1건일 때는 list가 아닌 dict 단일 객체로 옴.
    #    예) 결과 1건이면 "item": {...}, 여러 건이면 "item": [{...}, {...}]
    # field_validator(mode="before")로 Pydantic 타입 체크 전에 미리 list로 변환.
    @field_validator("item", mode="before")
    @classmethod
    def _coerce_to_list(cls, v):
        if v is None or v == "":      # 결과가 없을 때 빈 문자열로 오는 경우도 있음
            return []
        if isinstance(v, dict):       # 1건 케이스 → list로 감싸기
            return [v]
        return v                      # 이미 list면 그대로


class Body(BaseModel, Generic[T]):
    """응답 바디. items + 페이지 메타 정보."""
    items: Items[T] = Items[T]()
    dataType: str | None = None       # "JSON"
    pageNo: int                       # 현재 페이지 번호
    numOfRows: int                    # 페이지당 건수
    totalCount: int                   # 전체 건수 (페이지네이션 종료 조건용)


class Response_(BaseModel, Generic[T]):
    """언더스코어 suffix는 파이썬 예약어 Response와 혼동 방지용."""
    header: Header
    body: Body[T]


class PublicDataResponse(BaseModel, Generic[T]):
    """공공데이터포털 표준 JSON 응답 루트.
    최상위 키가 'Response' (대문자 R)인 점에 주의 — 다른 API는 'response'일 수 있음.
    """
    Response: Response_[T]


# ── 도메인별 Item 모델 ────────────────────────────────────────
# 각 API의 item 한 건이 어떤 필드를 갖는지 선언. 여기만 추가하면 fetch에서 재사용 가능.

class BizItem(BaseModel):
    """정산사업자 조회 API item (마스터 테이블의 소스).

    정산사업자 = 교통카드 요금 정산을 수행하는 기관.
    예: 티머니, 마이비, 이동의즐거움 등.
    """
    clcln_bzmn_id: str     # 2자리 ID (예: "03", "08")
    clcln_bzmn_nm: str     # 사업자명 (예: "티머니")


class TrfcMnsItem(BaseModel):
    """교통수단 조회 API item.
    정산사업자마다 자신이 다루는 교통수단 코드 체계를 가짐 → 복합키.
    """
    clcln_bzmn_id: str
    clcln_bzmn_trfc_mns_cd: str    # 교통수단 코드 (사업자 내 유일)
    clcln_bzmn_trfc_mns_nm: str    # 예: "간선버스(115)-기본요금"


class CardTypeItem(BaseModel):
    """카드 유형 조회 API item.
    같은 카드 코드라도 정산사업자에 따라 다른 의미 → 복합키 (biz_id + card_cd).
    """
    clcln_bzmn_id: str
    card_se_cd: str    # 카드 구분 코드 ("0"=현금, "1"=구선불카드, ...)
    card_se_nm: str    # 카드 구분명


class UserTypeItem(BaseModel):
    """이용자 유형 조회 API item.
    정산사업자 의존성 없음 → 단일 PK.
    예: 일반(성인), 청소년, 어린이, 경로(노인) 등.
    """
    users_type_cd: str
    users_type_nm: str


print("Pydantic 모델 로드 완료")

Pydantic 모델 로드 완료


In [33]:
# ============================================================
# 셀 2 - 공공데이터 API 페이지네이션 fetch (타입 안전)
# ============================================================
# 호출 규약: fetch_all_pages(url, ItemModel) → list[ItemModel]
# - item_model만 바꿔서 모든 공공데이터 표준 API에 재사용
# - 내부적으로 PublicDataResponse[item_model]로 Pydantic 검증
# - totalCount에 도달할 때까지 자동으로 페이지 순회


def fetch_all_pages(
    url: str,
    item_model: type[T],                  # 어떤 item 모델로 파싱할지
    extra_params: dict | None = None,     # 엔드포인트마다 필요한 추가 파라미터 (월별 필터 등)
    num_rows: int = 1000,                 # 한 페이지당 가져올 건수. 공공데이터 상한이 보통 1000
) -> list[T]:
    """공공데이터 표준 응답 API에서 전체 페이지를 수집해 검증된 item 리스트 반환."""

    # 공통 파라미터: 모든 요청에 공통으로 붙는 값
    # **extra_params 로 호출자가 추가 필터를 덮어쓸 수 있음
    params_base = {
        "serviceKey": SERVICE_KEY,
        "numOfRows": num_rows,
        "dataType": "JSON",
        **(extra_params or {}),
    }

    all_items: list[T] = []    # 모든 페이지에서 모은 검증된 item들
    page = 1

    while True:
        # 페이지 번호만 매 반복마다 갱신하여 요청
        res = requests.get(url, params={**params_base, "pageNo": page}, timeout=30)
        res.raise_for_status()  # HTTP 4xx/5xx면 여기서 예외 발생

        # Pydantic 검증: 스키마 불일치면 ValidationError 발생 (조기 에러 감지)
        # PublicDataResponse[item_model] → 런타임에 제네릭 타입 파라미터 지정
        parsed = PublicDataResponse[item_model].model_validate(res.json())

        # 성공 코드 확인. API마다 "00" 또는 "200"을 성공으로 씀.
        header = parsed.Response.header
        if header.resultCode not in ("00", "200"):
            raise RuntimeError(f"API error: {header.resultCode} {header.resultMsg}")

        # 검증된 item 리스트 추출 → 누적
        body = parsed.Response.body
        items = body.items.item
        all_items.extend(items)
        print(f"  page {page}: +{len(items)}건 (누적 {len(all_items)}/{body.totalCount})")

        # 종료 조건:
        #   1) 지금까지 모은 건수가 totalCount에 도달했거나
        #   2) 이번 페이지가 빈 배열 (방어적 체크, 정상 API면 1번에서 먼저 종료됨)
        if len(all_items) >= body.totalCount or not items:
            break

        page += 1
        time.sleep(0.2)   # API rate limit 방지. 너무 빠르면 500 에러 가능성

    return all_items

In [34]:
# ============================================================
# 셀 3 - 교통수단 조회 전체 페이지 수집
# ============================================================
# 엔드포인트: 정산사업자별 교통수단 코드/명칭 매핑
# 예상 규모: ~2000건 (정산사업자 x 교통수단 조합)

TRFC_URL = "https://apis.data.go.kr/1613000/PublicTransportationMode/getPublicTransportationMode"

# fetch_all_pages가 모든 페이지 자동 순회 → TrfcMnsItem 인스턴스 리스트 반환
trfc_items = fetch_all_pages(TRFC_URL, TrfcMnsItem)

# Pydantic 객체 → dict → DataFrame (DuckDB 적재에 편함)
trfc_df = pd.DataFrame([it.model_dump() for it in trfc_items])

print(f"\n총 {len(trfc_df)}건 수집")
print(trfc_df.head())

  page 1: +1000건 (누적 1000/1959)
  page 2: +959건 (누적 1959/1959)

총 1959건 수집
  clcln_bzmn_id clcln_bzmn_trfc_mns_cd clcln_bzmn_trfc_mns_nm
0            03                    105         마을버스(105)-기본요금
1            03                    110        주간선버스(110)-기본요금
2            03                    115         간선버스(115)-기본요금
3            03                    120         지선버스(120)-기본요금
4            03                    121         지선버스(121)-기본요금


In [ ]:
# ============================================================
# 셀 4 - DuckDB에 적재 (trfc_mns 테이블, 복합 PK)
# ============================================================
# db_open()으로 짧게 연결 → 작업 → 자동 close (예외 시에도 보장)

with db_open() as con:
    con.execute("""
    CREATE TABLE IF NOT EXISTS trfc_mns (
        clcln_bzmn_id            TEXT,  -- 정산사업자 ID (예: "03"=마이비)
        clcln_bzmn_trfc_mns_cd   TEXT,  -- 교통수단 코드 (예: "115"=간선버스)
        clcln_bzmn_trfc_mns_nm   TEXT,  -- 교통수단명
        PRIMARY KEY (clcln_bzmn_id, clcln_bzmn_trfc_mns_cd)
    )
    """)

    # 메타성 테이블이라 UPSERT 대신 전체 교체가 단순함.
    con.execute("DELETE FROM trfc_mns")

    con.register("trfc_df_view", trfc_df)
    con.execute("""
    INSERT INTO trfc_mns
    SELECT clcln_bzmn_id, clcln_bzmn_trfc_mns_cd, clcln_bzmn_trfc_mns_nm
    FROM trfc_df_view
    """)
    con.unregister("trfc_df_view")

    cnt = con.execute("SELECT COUNT(*) FROM trfc_mns").fetchone()[0]

print(f"적재 완료: {cnt}건")

In [36]:
# ============================================================
# 셀 5 - 카드 유형 조회 전체 페이지 수집
# ============================================================
# 정산사업자별로 취급하는 카드/결제수단 코드 목록.
# "현금", "구선불카드", "후불카드" 등.

CARD_URL = "https://apis.data.go.kr/1613000/PublicTransportationCardType/getPublicTransportationCardType"

card_items = fetch_all_pages(CARD_URL, CardTypeItem)
card_df = pd.DataFrame([it.model_dump() for it in card_items])

print(f"\n총 {len(card_df)}건 수집")
print(card_df.head())

  page 1: +105건 (누적 105/105)

총 105건 수집
  clcln_bzmn_id card_se_cd card_se_nm
0            03          0         현금
1            03          1      구선불카드
2            03          2       후불카드
3            03          3    고급형선불카드
4            03          4      신후불카드


In [ ]:
# ============================================================
# 셀 6 - DuckDB에 적재 (card_type 테이블, 복합 PK)
# ============================================================
with db_open() as con:
    con.execute("""
    CREATE TABLE IF NOT EXISTS card_type (
        clcln_bzmn_id  TEXT,  -- 정산사업자 ID
        card_se_cd     TEXT,  -- 카드 구분 코드
        card_se_nm     TEXT,  -- 카드 구분명
        PRIMARY KEY (clcln_bzmn_id, card_se_cd)
    )
    """)

    con.execute("DELETE FROM card_type")
    con.register("card_df_view", card_df)
    con.execute("""
    INSERT INTO card_type
    SELECT clcln_bzmn_id, card_se_cd, card_se_nm
    FROM card_df_view
    """)
    con.unregister("card_df_view")

    cnt = con.execute("SELECT COUNT(*) FROM card_type").fetchone()[0]

print(f"적재 완료: {cnt}건")

In [38]:
# ============================================================
# 셀 7 - 정산사업자(biz) 조회 전체 페이지 수집
# ============================================================
# 마스터 테이블의 소스. ID(clcln_bzmn_id) → 사업자명(clcln_bzmn_nm) 매핑.
# trfc_mns, card_type이 참조하는 루트.

BIZ_URL = "https://apis.data.go.kr/1613000/PublicTransportationFareSettlementOperator/getPublicTransportationFareSettlementOperator"

biz_items = fetch_all_pages(BIZ_URL, BizItem)
biz_df = pd.DataFrame([it.model_dump() for it in biz_items])

print(f"\n총 {len(biz_df)}건 수집")
print(biz_df.head(20))

  page 1: +10건 (누적 10/10)

총 10건 수집
  clcln_bzmn_id  clcln_bzmn_nm
0            03            마이비
1            08            티머니
2            11         이동의즐거움
3            14        iM유페이먼트
4            15          한페이시스
5            17          글로벌캐쉬
6            18  전국버스운송사업조합연합회
7            19    전국고속버스조합연합회
8            20    한국철도공사(동해선)
9            30  한국철도공사(발권데이터)


In [ ]:
# ============================================================
# 셀 8 - DuckDB에 적재 (biz 테이블, 단일 PK — 마스터 테이블)
# ============================================================
with db_open() as con:
    con.execute("""
    CREATE TABLE IF NOT EXISTS biz (
        clcln_bzmn_id  TEXT PRIMARY KEY,  -- 정산사업자 ID
        clcln_bzmn_nm  TEXT               -- 정산사업자명 (예: 마이비, 티머니, ...)
    )
    """)

    con.execute("DELETE FROM biz")
    con.register("biz_df_view", biz_df)
    con.execute("""
    INSERT INTO biz
    SELECT clcln_bzmn_id, clcln_bzmn_nm
    FROM biz_df_view
    """)
    con.unregister("biz_df_view")

    cnt = con.execute("SELECT COUNT(*) FROM biz").fetchone()[0]
    print(f"적재 완료: {cnt}건")

    # JOIN 동작 검증 쿼리 (같은 with 블록에서 이어서 실행)
    print("\n=== 정산사업자별 교통수단/카드유형 개수 ===")
    print(con.execute("""
        SELECT b.clcln_bzmn_id, b.clcln_bzmn_nm,
               COUNT(DISTINCT t.clcln_bzmn_trfc_mns_cd) AS trfc_cnt,
               COUNT(DISTINCT c.card_se_cd)            AS card_cnt
        FROM biz b
        LEFT JOIN trfc_mns  t USING (clcln_bzmn_id)
        LEFT JOIN card_type c USING (clcln_bzmn_id)
        GROUP BY 1, 2
        ORDER BY trfc_cnt DESC
    """).df())

In [40]:
# ============================================================
# 셀 9 - 이용자 유형(user_type) 조회 전체 페이지 수집
# ============================================================
# 정산사업자 의존성 없는 전역 차원 테이블.
# 예: "001"=일반(성인), "002"=청소년, "003"=어린이, "004"=경로(노인) 등
# 이 프로젝트에서 '노인' 관련 집계할 때 핵심 차원.

USER_TYPE_URL = "https://apis.data.go.kr/1613000/PublicTransportationUserType/getPublicTransportationUserType"

user_type_items = fetch_all_pages(USER_TYPE_URL, UserTypeItem)
user_type_df = pd.DataFrame([it.model_dump() for it in user_type_items])

print(f"\n총 {len(user_type_df)}건 수집")
print(user_type_df)

  page 1: +8건 (누적 8/8)

총 8건 수집
  users_type_cd users_type_nm
0            01           일반인
1            02           어린이
2            03           청소년
3            04            경로
4            05           장애인
5            06         국가유공자
6            07           외국인
7            08            기타


In [ ]:
# ============================================================
# 셀 10 - DuckDB에 적재 (user_type 테이블, 단일 PK)
# ============================================================
with db_open() as con:
    con.execute("""
    CREATE TABLE IF NOT EXISTS user_type (
        users_type_cd  TEXT PRIMARY KEY,  -- 이용자 유형 코드
        users_type_nm  TEXT               -- 이용자 유형명 (일반/청소년/어린이/경로 등)
    )
    """)

    con.execute("DELETE FROM user_type")
    con.register("user_type_df_view", user_type_df)
    con.execute("""
    INSERT INTO user_type
    SELECT users_type_cd, users_type_nm
    FROM user_type_df_view
    """)
    con.unregister("user_type_df_view")

    cnt = con.execute("SELECT COUNT(*) FROM user_type").fetchone()[0]
    print(f"적재 완료: {cnt}건")
    print(con.execute("SELECT * FROM user_type ORDER BY users_type_cd").df())

In [ ]:
# ============================================================
# 셀 11 - 검증
# ============================================================
# read_only=True → 다른 커널이 쓰기 중이어도 읽기 가능 (충돌 없음)

with db_open(read_only=True) as con:
    print("=== trfc_mns: 정산사업자별 교통수단 수 ===")
    print(con.execute("""
        SELECT clcln_bzmn_id, COUNT(*) AS cnt
        FROM trfc_mns
        GROUP BY 1 ORDER BY 2 DESC
    """).df())

    print("\n=== card_type: 정산사업자별 카드유형 수 ===")
    print(con.execute("""
        SELECT clcln_bzmn_id, COUNT(*) AS cnt
        FROM card_type
        GROUP BY 1 ORDER BY 2 DESC
    """).df())

    print("\n=== card_type 샘플 15건 ===")
    print(con.execute("SELECT * FROM card_type LIMIT 15").df())

    print("\n=== user_type 전체 ===")
    print(con.execute("SELECT * FROM user_type ORDER BY users_type_cd").df())